In [ ]:
# Capability / introspection: probe the runtime SIMD dispatch surface.
import numkong

public_names = [n for n in dir(numkong) if not n.startswith('_')]
print(f"numkong public names ({len(public_names)}):")
for name in public_names:
    obj = getattr(numkong, name)
    print(f"  {name}: {type(obj).__name__}")

assert len(public_names) > 0, "numkong should expose at least one public symbol"


In [ ]:
# Basic numpy interop: feed numkong typical numpy float32 arrays.
import numpy as np
import numkong

rng = np.random.default_rng(42)
a = rng.standard_normal(1024).astype(np.float32)
b = rng.standard_normal(1024).astype(np.float32)

print(f"a.dtype={a.dtype} a.shape={a.shape}")
print(f"b.dtype={b.dtype} b.shape={b.shape}")

# numpy is the reference: numkong's dot product must agree with it.
ref = float(np.dot(a, b))
got = float(numkong.dot(a, b))
print(f"numpy dot reference: {ref:.6f}")
print(f"numkong.dot:         {got:.6f}")

assert np.isfinite(ref), "reference dot product must be finite"
assert np.isclose(got, ref, rtol=1e-3, atol=1e-4), (got, ref)


In [ ]:
# Compare numkong's vector kernels with the equivalent numpy operations.
import numpy as np
import numkong

rng = np.random.default_rng(0)
x = rng.standard_normal(256).astype(np.float32)
y = rng.standard_normal(256).astype(np.float32)

# Reductions to a scalar
scalar_checks = {
    'dot': (numkong.dot(x, y), np.dot(x, y)),
    'inner': (numkong.inner(x, y), np.inner(x, y)),
    'sqeuclidean': (numkong.sqeuclidean(x, y), np.sum((x - y) ** 2)),
    'euclidean': (numkong.euclidean(x, y), np.linalg.norm(x - y)),
}
for name, (got, want) in scalar_checks.items():
    print(f"  numkong.{name}: {float(got):.6f} (numpy {float(want):.6f})")
    assert np.isclose(float(got), float(want), rtol=1e-3, atol=1e-4), (name, got, want)

# Element-wise kernels returning a vector
vector_checks = {
    'add': (numkong.add(x, y), x + y),
    'multiply': (numkong.multiply(x, y), x * y),
}
for name, (got, want) in vector_checks.items():
    got = np.asarray(got)
    print(f"  numkong.{name}: shape={got.shape} dtype={got.dtype}")
    assert got.shape == want.shape, (name, got.shape)
    assert np.allclose(got, want, rtol=1e-5, atol=1e-6), name


In [ ]:
import sys
import numkong

# Find any loaded extension modules that originated from numkong.
loaded = [name for name in sys.modules if name.startswith('numkong')]
print(f"Loaded numkong-related modules: {loaded}")

ext_files = []
for name in loaded:
    mod = sys.modules[name]
    f = getattr(mod, '__file__', None)
    if f and (f.endswith('.so') or '.cpython-' in f):
        ext_files.append(f)

print(f"C-extension files: {ext_files}")
assert len(loaded) >= 1, "numkong should register at least one module in sys.modules"

In [ ]:
# Final pass signal: re-import is idempotent and the module object is stable.
import importlib
import numkong

first_id = id(numkong)
reloaded = importlib.import_module('numkong')
second_id = id(reloaded)

print(f"first import id  = {first_id}")
print(f"second import id = {second_id}")

assert first_id == second_id, "importlib.import_module must return the cached module object"
print("\nnumkong smoke test PASSED")
